In [ ]:
"""
md_to_dom.py

Converts an OCR-generated markdown file (headings + GFM pipe tables --
the kind produced by tools like Marker, MinerU, Textract-to-md, etc.) into
a Logical DOM tree:

    Document -> Section -> Table -> Row -> Cell

No bounding boxes are assumed or required. Structure is derived from:
  - markdown heading lines (#, ##, ###, ... and bold-only pseudo-headings)
  - GFM pipe-table syntax (| a | b | ... | with a --- separator row)
  - row "marker" cells (I, 1, (a), (i) ...) used for parent-child nesting,
    matching the numbering convention used in Indian financial statements
  - a "Note" column (if present) used to cross-reference rows to the
    detailed note tables that appear later in the same document

This is intentionally a STRUCTURE parser, not a data-cleaning tool. Real
OCR->markdown output is messy (misaligned headers, reversed columns,
stray characters). This script organizes that mess into a navigable tree;
it does not attempt to correct OCR errors.

Usage:
    python md_to_dom.py input.md -o output.json
    python md_to_dom.py input.md --print-tree
"""

from __future__ import annotations

import argparse
import json
import re
from dataclasses import dataclass, field, asdict
from typing import Optional


# --------------------------------------------------------------------------
# Data model
# --------------------------------------------------------------------------

@dataclass
class Cell:
    col_index: int
    header: str
    value: str

@dataclass
class Row:
    row_index: int
    marker: str                 # raw leading cell, e.g. "I", "1", "(a)", "(i)"
    label: str                  # best-guess row label (the "Particulars" text)
    level: Optional[int]        # nesting depth inferred from marker, None if unclassified
    note_ref: Optional[str]     # value from a "Note" column, if the table has one
    cells: list[Cell] = field(default_factory=list)
    children: list["Row"] = field(default_factory=list)

    def to_dict(self):
        d = asdict(self)
        return d

@dataclass
class Table:
    table_index: int
    title: Optional[str]        # nearest heading / section title
    note_number: Optional[str]  # parsed from title if it looks like "Note - 3a" / "NOTE 4"
    headers: list[str]
    rows: list[Row] = field(default_factory=list)

    def to_dict(self):
        return {
            "table_index": self.table_index,
            "title": self.title,
            "note_number": self.note_number,
            "headers": self.headers,
            "rows": [r.to_dict() for r in self.rows],
        }

@dataclass
class Section:
    section_index: int
    heading: str
    level: int                  # markdown heading level, 1-6
    paragraphs: list[str] = field(default_factory=list)
    tables: list[Table] = field(default_factory=list)

    def to_dict(self):
        return {
            "section_index": self.section_index,
            "heading": self.heading,
            "level": self.level,
            "paragraphs": self.paragraphs,
            "tables": [t.to_dict() for t in self.tables],
        }

@dataclass
class Document:
    source_file: str
    sections: list[Section] = field(default_factory=list)

    def to_dict(self):
        return {
            "source_file": self.source_file,
            "sections": [s.to_dict() for s in self.sections],
        }

    # ---- convenience lookups, used by the resolver / navigator later ----

    def find_table_by_note(self, note_number: str) -> Optional[Table]:
        """Find the detail table for a given note number, e.g. '4' or '3a'."""
        for section in self.sections:
            for table in section.tables:
                if table.note_number == note_number:
                    return table
        return None

    def iter_tables(self):
        for section in self.sections:
            for table in section.tables:
                yield section, table


# --------------------------------------------------------------------------
# Parsing helpers
# --------------------------------------------------------------------------

HEADING_RE = re.compile(r'^(#{1,6})\s+(.*\S)\s*$')
BOLD_ONLY_RE = re.compile(r'^\*\*(.+?)\*\*$')
TABLE_ROW_RE = re.compile(r'^\|.*\|$')
TABLE_SEP_RE = re.compile(r'^\|[\s\-:|]+\|$')
NOTE_HEADING_RE = re.compile(r'note\s*[-:]?\s*(\d+[a-z]?)\b', re.IGNORECASE)
YEAR_RE = re.compile(r'(19|20)\d{2}')

# Row marker patterns, in the order we test them (most specific first),
# each mapped to a nesting level. This matches the Roman-numeral / digit /
# (a) / (i) convention used across Indian LLP & company financial statements.
MARKER_LEVELS = [
    (re.compile(r'^[IVXLCDM]+$', re.IGNORECASE), 0),   # I, II, III, IV ...
    (re.compile(r'^\d+$'), 1),                          # 1, 2, 3 ...
    (re.compile(r'^\([a-z]\)$', re.IGNORECASE), 2),     # (a), (b), (c) ...
    (re.compile(r'^\([ivxlcdm]+\)$', re.IGNORECASE), 3),# (i), (ii), (iii) ...
]


def classify_marker(marker: str) -> Optional[int]:
    marker = marker.strip()
    if not marker:
        return None
    for pattern, level in MARKER_LEVELS:
        if pattern.match(marker):
            return level
    return None


def split_row(line: str) -> list[str]:
    """Split a markdown table row into stripped cell strings."""
    inner = line.strip()
    if inner.startswith('|'):
        inner = inner[1:]
    if inner.endswith('|'):
        inner = inner[:-1]
    return [c.strip() for c in inner.split('|')]


def looks_like_header_continuation(cells: list[str]) -> bool:
    """
    Detects the common OCR-table quirk where the 'real' column headers are
    split across two rows, e.g.:
        | Particulars | Note | As at | As at |
        |--------------------------------------|
        | $           |      | March 31, 2025  | March 31, 2024 |   <- this row
        | (a) | Trade payables | 4 | 12,10,692 | 84,740 |            <- real data starts here
    Heuristic: the row contains a year (2024/2025 etc.) in a non-first cell,
    and its first one or two cells are empty or a single stray character
    (not a real marker/label).
    """
    if not cells:
        return False
    has_year = any(YEAR_RE.search(c) for c in cells)
    if not has_year:
        return False
    lead = cells[0].strip()
    lead_is_noise = (lead == '' or len(lead) <= 2 and not lead.isdigit())
    return lead_is_noise


def merge_headers(main_headers: list[str], sub_row: list[str]) -> list[str]:
    merged = []
    for i in range(max(len(main_headers), len(sub_row))):
        h = main_headers[i] if i < len(main_headers) else ''
        s = sub_row[i] if i < len(sub_row) else ''
        parts = [p for p in (h.strip(), s.strip()) if p]
        merged.append(' '.join(parts))
    return merged


def find_note_column(headers: list[str]) -> Optional[int]:
    for i, h in enumerate(headers):
        if re.search(r'\bnote\b', h, re.IGNORECASE):
            return i
    return None


def build_row_tree(rows: list[Row]) -> list[Row]:
    """
    Nest rows into a parent/child tree based on inferred marker level.
    Rows with no classifiable marker (None level) attach as a child of the
    most recent row (continuation lines, sub-items without a clean marker).
    Returns the list of top-level rows; every row's .children is populated.
    """
    roots: list[Row] = []
    stack: list[Row] = []  # stack of (row) ordered by level, shallow -> deep

    for row in rows:
        level = row.level
        if level is None:
            # attach as a child of the deepest currently open row, if any
            if stack:
                stack[-1].children.append(row)
            else:
                roots.append(row)
            continue

        # pop stack until we find a parent shallower than this row's level
        while stack and stack[-1].level is not None and stack[-1].level >= level:
            stack.pop()

        if stack:
            stack[-1].children.append(row)
        else:
            roots.append(row)

        stack.append(row)

    return roots


# --------------------------------------------------------------------------
# Main parser
# --------------------------------------------------------------------------

def parse_markdown(text: str, source_file: str = "") -> Document:
    lines = text.splitlines()
    doc = Document(source_file=source_file)

    # a synthetic "preamble" section holds any content before the first heading
    current_section = Section(section_index=0, heading="(preamble)", level=0)
    doc.sections.append(current_section)

    section_counter = 0
    table_counter = 0
    i = 0
    n = len(lines)

    while i < n:
        line = lines[i]
        stripped = line.strip()

        # --- heading? ---
        m = HEADING_RE.match(stripped)
        bold_m = BOLD_ONLY_RE.match(stripped) if not m else None
        if m or bold_m:
            section_counter += 1
            if m:
                level = len(m.group(1))
                heading_text = m.group(2).strip('* ').strip()
            else:
                level = 6  # treat bold-only pseudo-headings as a deep level
                heading_text = bold_m.group(1).strip()
            current_section = Section(section_index=section_counter, heading=heading_text, level=level)
            doc.sections.append(current_section)
            i += 1
            continue

        # --- table? ---
        if TABLE_ROW_RE.match(stripped):
            table_lines = []
            while i < n and TABLE_ROW_RE.match(lines[i].strip()):
                table_lines.append(lines[i].strip())
                i += 1

            if len(table_lines) >= 2 and TABLE_SEP_RE.match(table_lines[1]):
                header_cells = split_row(table_lines[0])
                data_lines = table_lines[2:]
            else:
                # no proper separator row found; treat first line as header anyway
                header_cells = split_row(table_lines[0])
                data_lines = table_lines[1:]

            # merge a header-continuation row into the headers, if present
            if data_lines:
                first_data_cells = split_row(data_lines[0])
                if looks_like_header_continuation(first_data_cells):
                    header_cells = merge_headers(header_cells, first_data_cells)
                    data_lines = data_lines[1:]

            note_col = find_note_column(header_cells)

            table_counter += 1
            note_number = None
            nm = NOTE_HEADING_RE.search(current_section.heading)
            if nm:
                note_number = nm.group(1)

            table = Table(
                table_index=table_counter,
                title=current_section.heading,
                note_number=note_number,
                headers=header_cells,
            )

            flat_rows: list[Row] = []
            for r_idx, dline in enumerate(data_lines):
                cells_raw = split_row(dline)
                marker = cells_raw[0] if cells_raw else ""
                level_guess = classify_marker(marker)

                # best-guess label: first non-empty cell after the marker,
                # falling back to the marker itself
                label = ""
                for c in cells_raw[1:]:
                    if c.strip():
                        label = c.strip()
                        break
                if not label:
                    label = marker

                note_ref = None
                if note_col is not None and note_col < len(cells_raw):
                    val = cells_raw[note_col].strip()
                    if re.match(r'^\d+[a-z]?$', val, re.IGNORECASE):
                        note_ref = val

                cells = [
                    Cell(col_index=ci, header=(header_cells[ci] if ci < len(header_cells) else ""), value=cv)
                    for ci, cv in enumerate(cells_raw)
                ]

                row = Row(
                    row_index=r_idx,
                    marker=marker,
                    label=label,
                    level=level_guess,
                    note_ref=note_ref,
                    cells=cells,
                )
                flat_rows.append(row)

            table.rows = build_row_tree(flat_rows)
            current_section.tables.append(table)
            continue

        # --- plain paragraph text ---
        if stripped and not stripped.startswith('!['):  # skip image placeholders
            current_section.paragraphs.append(stripped)

        i += 1

    return doc


# --------------------------------------------------------------------------
# CLI
# --------------------------------------------------------------------------

def print_tree(doc: Document, max_rows_per_table: int = 8):
    for section in doc.sections:
        if not section.tables and not section.paragraphs:
            continue
        print(f"\n{'#' * max(section.level, 1)} [{section.section_index}] {section.heading}")
        for table in section.tables:
            note_str = f" (Note {table.note_number})" if table.note_number else ""
            print(f"  Table[{table.table_index}]{note_str}  headers={table.headers}")

            def _print_row(row: Row, indent: int):
                pad = "    " + "  " * indent
                note_str = f"  [note_ref={row.note_ref}]" if row.note_ref else ""
                print(f"{pad}- ({row.marker}) {row.label}{note_str}")
                for child in row.children:
                    _print_row(child, indent + 1)

            for row in table.rows[:max_rows_per_table]:
                _print_row(row, 0)
            if len(table.rows) > max_rows_per_table:
                print(f"    ... ({len(table.rows) - max_rows_per_table} more rows)")


def main():
    ap = argparse.ArgumentParser(description="Convert OCR markdown output into a Logical DOM tree.")
    ap.add_argument("input", help="Path to the OCR .md file")
    ap.add_argument("-o", "--output", help="Path to write Logical_DOM.json", default=None)
    ap.add_argument("--print-tree", action="store_true", help="Pretty-print the resulting tree to stdout")
    args = ap.parse_args()

    with open(args.input, "r", encoding="utf-8") as f:
        text = f.read()

    doc = parse_markdown(text, source_file=args.input)

    if args.print_tree:
        print_tree(doc)

    if args.output:
        with open(args.output, "w", encoding="utf-8") as f:
            json.dump(doc.to_dict(), f, indent=2, ensure_ascii=False)
        print(f"\nWrote {args.output}")


if __name__ == "__main__":
    main()